# Project 6: learn a simple product RAG evaluation

We have **20 products** and **78 supplied questions**. Our learning loop is:

**Retrieve evidence → generate an answer → judge it → inspect one failure → improve → compare.**

Steps 1–6 introduce the data, retrieval and answer scores. Steps 7–10 follow one question through a small improvement. We use the same custom Gemini judge throughout.

Embeddings run locally on CPU. Answers use Groq; judging uses Gemini. This notebook opens saved real results by default. The optional rerun makes calls only for the one teaching question.

## 1. Setup and helper roles

Select the shared root `.venv` kernel in VS Code. Open this notebook from Project 6 or the repository root, and restart the kernel after editing imported files.

- `services/prepare_documents.py` loads the catalog and creates the vector store. Run preparation separately when rebuilding.
- `models/` loads embeddings and the stored vectors, and calls the models.
- `services/retriever.py` selects products; `services/rag.py` sends their text and your question to Groq.
- `tools/retrieval_eval.py` calculates retrieval scores; `tools/quality_eval.py` asks Gemini to judge answers.
- `tools/results.py` reads the saved JSON reports used below. Reading them makes no model calls.

The golden answers are for evaluation. They are not sent to the answering model. Supplied references and mappings remain pending human review.

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd

project = Path.cwd()
if project.name != '6_product_rag_eval':
    project = project / '6_product_rag_eval'
project = project.resolve()
assert project.is_dir(), 'Open from Project 6 or the repository root.'
sys.path.insert(0, str(project))

from models.settings import load_catalog
from tools.retrieval_eval import load_cases
from tools.results import load_artifact, load_first_baselines, quality_table

products = load_catalog()
cases = load_cases()
print('Products:', len(products))
print('Golden questions:', len(cases))
run = json.loads((project / 'docs/evalresults/latest_run.json').read_text(encoding='utf-8'))
display(pd.Series({key: run[key] for key in ['run_id', 'generator', 'judge', 'embedding_model', 'human_review']}))
display(pd.DataFrame(cases)[['id', 'test_type', 'question', 'expected_retrieval_scope', 'expected_product_ids']].head())

Products: 20
Golden questions: 78


run_id                     20261016T140244531919Z
generator                     openai/gpt-oss-120b
judge                       gemini-3.5-flash-lite
embedding_model    nomic-ai/modernbert-embed-base
human_review                              pending
dtype: str

,id,test_type,question,expected_retrieval_scope,expected_product_ids
0,Q1,Local product lookup,What is the manufacturer and starting price of...,HCL-101,[HCL-101]
1,Q2,Leaf / error-code lookup,"For AeroTherm X1, what does error code CLM-E10...",HCL-101,[HCL-101]
2,Q3,Variant lookup,Which variant ID and price correspond to the S...,HCL-101-S,[HCL-101]
3,Q4,Local product lookup,What is the manufacturer and starting price of...,HCL-102,[HCL-102]
4,Q5,Leaf / error-code lookup,"For BreezeSense V2, what does error code CLM-E...",HCL-102,[HCL-102]


## 2. Preserve the first 40-question experiment

The original product/error-code baseline retrieved three products. Its recall was 1.0 and precision 0.333. Reducing k to one retained recall and raised precision to 1.0. These saved reports are preserved.

Each of these 40 questions needs one product. This result does not establish that k=1 works for category or catalog-wide questions.

In [2]:
first_reports = load_first_baselines()
display(pd.DataFrame([{
    'experiment': r['experiment'], 'questions': r['questions_evaluated'],
    'k': r['k'], 'recall': r['recall_at_k'], 'precision': r['precision_at_k']
} for r in first_reports]))

,experiment,questions,k,recall,precision
0,baseline_k3,40,3,1.0,0.333333
1,candidate_k1,40,1,1.0,1.000000


## 3. Evaluate retrieval on all 78 questions

Variant IDs map to their parent product because one stored document contains a complete product and its variants. Category and subcategory references include every product in that group. Global and absence questions use conservative catalog-wide evidence sets; other sufficient evidence sets can exist.

Compare plain semantic k=1 retrieval with a small **catalog-aware candidate**. The candidate uses catalog metadata for group requests and retrieves the complete catalog for broad requests. It reads no reference answers or golden IDs. k=1 still applies to its semantic fallback.

Product-ID recall measures evidence coverage; it does not measure how much of a product's text is relevant. Precision in this candidate is not a fixed-k precision because broad routes return more than one document.

In [3]:
semantic = load_artifact('retrieval_semantic')
scoped = load_artifact('retrieval_catalog')
display(pd.DataFrame([{
    'retriever': r['mode'], 'questions': r['questions_evaluated'],
    'recall': r['recall_at_k'], 'precision': r['precision_at_k'],
    'all_reference_evidence_found': r['full_evidence_rate']
} for r in [semantic, scoped]]))
display(pd.DataFrame(scoped['results']).groupby('test_type')[['recall', 'precision']].mean())

,retriever,questions,recall,precision,all_reference_evidence_found
0,semantic,78,0.841132,0.987179,0.769231
1,catalog,78,1.000000,1.000000,1.000000


,recall,precision
test_type,,
Ambiguity test,1.0,1.0
Category aggregation,1.0,1.0
Cross-category comparison,1.0,1.0
Global reasoning,1.0,1.0
Leaf / error-code lookup,1.0,1.0
Local product lookup,1.0,1.0
Missing-child test,1.0,1.0
Negative retrieval,1.0,1.0
Subcategory aggregation,1.0,1.0


## 4. Inspect the generator separately

To inspect generation alone, select the source products using the reference product IDs, then give only the question and those source documents to the generator. This removes retrieval mistakes from this small check.

The generator still does not receive the expected answer. Read the three saved answers below. We introduce our one judging method in Step 6.

In [4]:
isolation = load_artifact('generator_isolation', outputs=True)
display(pd.DataFrame([
    {'id': row['id'], 'status': row['status'], 'answer': row['answer']}
    for row in isolation
]))

,id,status,answer
0,Q1,answered,The AeroTherm X1 is manufactured by HelioWorks...
1,Q3,answered,The Standard variant has ID HCL-101-S and cost...
2,Q61,answered,Home Appliances products:\n- BrewCraft Duo (AP...


## 5. Inspect the complete RAG

The complete application retrieves source products and generates an answer. It already answered all 78 questions in the saved run.

The table shows which retrieval route was used, which citations were returned, and what the model answered. The reference answers stay outside the generator's inputs.

In [5]:
answers = load_artifact('candidate_answers', outputs=True)
judgments = load_artifact('candidate_judgments')
summary = load_artifact('candidate_summary')
print('Saved answers:', len(answers), '| Judged answers:', summary['judged'])
display(pd.DataFrame([{'id': r['id'], 'route': r['route'], 'status': r['status'], 'citations': r['citations'], 'answer': r['answer']} for r in answers]).head(8))

Saved answers: 78 | Judged answers: 78


,id,route,status,citations,answer
0,Q1,semantic,answered,[HCL-101],The AeroTherm X1 is manufactured by HelioWorks...
1,Q2,semantic,answered,[HCL-101],Error code CLM‑E101 means “Sensor unavailable”...
2,Q3,semantic,answered,[HCL-101],The Standard variant has ID HCL-101-S and cost...
3,Q4,semantic,answered,[HCL-102],The BreezeSense V2 is manufactured by HelioWor...
4,Q5,semantic,answered,[HCL-102],Error code CLM‑E118 for the BreezeSense V2 ind...
5,Q6,semantic,answered,[HCL-102],The Base variant has ID HCL-102-B and costs $189.
6,Q7,semantic,answered,[HSC-201],The Sentinel Lock S4 is made by Aegis Entry Sy...
7,Q8,semantic,answered,[HSC-201],Error code SEC‑E101 means the bolt is obstruct...


## 6. Understand the answer scores

One Gemini judge reads the question, source context, expected answer and generated answer. For now, focus on six fields from its existing rubric:

| Score | Question it asks |
| --- | --- |
| Faithfulness | Are the claims supported by the retrieved text? |
| Answer relevance | Does the answer address the question? |
| Context relevance | Is the retrieved text useful for the question? |
| Correctness | Are the stated facts correct against the reference? |
| Completeness | Did it include every requested point? |
| Style | Is it clear and easy to read? |

These custom scores range from 0 to 1. Higher is better for these six fields. A judge can make mistakes, so we will now inspect one example ourselves. Our improvement example focuses on **completeness**.

In [6]:
score_names = ['faithfulness', 'answer_relevance', 'context_relevance',
               'correctness', 'completeness', 'style']
display(pd.Series({name: summary['quality'][name] for name in score_names},
                  name='Saved judge average'))
quality = pd.DataFrame(quality_table(answers, judgments))
display(quality[['id', 'correctness', 'completeness']].head(10))

faithfulness         0.993590
answer_relevance     0.997436
context_relevance    0.999359
correctness          0.974359
completeness         0.976923
style                1.000000
Name: Saved judge average, dtype: float64

,id,correctness,completeness
0,Q1,1.0,1.0
1,Q2,1.0,1.0
2,Q3,1.0,1.0
3,Q4,1.0,1.0
4,Q5,1.0,1.0
5,Q6,1.0,1.0
6,Q7,1.0,1.0
7,Q8,1.0,1.0
8,Q9,1.0,1.0
9,Q10,1.0,1.0


## 7. Read one answer from beginning to end

Question **Q61** asks for all Home Appliances products. We will use this same question and reference throughout Steps 7–10.

A baseline is simply the starting version. Its `mode="semantic"` setting retrieves the nearest product with `k=1`. The following cell loads its real saved answer and judge result; it does not run a new model call.

Read the question, expected answer and generated answer before looking at the score.

In [7]:
example_id = 'Q61'
example = next(case for case in cases if case['id'] == example_id)
baseline = next(row for row in load_artifact('baseline_answers', outputs=True)
                if row['id'] == example_id)
baseline_judge = next(row for row in load_artifact('baseline_judgments')
                      if row['id'] == example_id)

print('QUESTION:', example['question'])
print('\nEXPECTED:', example['expected_answer'])
print('\nBASELINE ANSWER:', baseline['answer'])

QUESTION: List all products under the Home Appliances category.

EXPECTED: BrewCraft Duo; FreshVault 12; CleanSweep R3; SteamMate S1; AirPure Cube; SleepLight Rhythm.

BASELINE ANSWER: FreshVault 12 (product ID APK-802) is a Home Appliances product.


## 8. Understand the failure

The reference asks for six products. The baseline retrieved one product and named that one product correctly. It had too little evidence to list all six.

This explains how **correctness can be high while completeness is low**. Read the source product IDs and the judge's completeness explanation below.

Our hypothesis: retrieving every product in the requested category should make the answer more complete. We will keep the question, reference, generator, prompt and judge the same.

In [8]:
print('Required product IDs:', example['expected_product_ids'])
print('Retrieved product IDs:', baseline['retrieved_ids'])
print('Correctness:', baseline_judge['data']['correctness'])
print('Completeness:', baseline_judge['data']['completeness'])
print('Judge explanation:', baseline_judge['data']['reasons']['completeness'])

Required product IDs: ['APK-801', 'APK-802', 'APC-901', 'APC-902', 'APW-1001', 'APW-1002']
Retrieved product IDs: ['APK-802']
Correctness: 1.0
Completeness: 0.17
Judge explanation: The expected answer lists six products, but only one is provided in the actual answer due to the limited context supplied.


## 9. Make one change: retrieve the category

The one changed setting is **`mode="semantic"` → `mode="catalog"`**. The generator and judge are unchanged.

In `services/retriever.py`, catalog mode notices the category name and selects all matching products from catalog metadata. It does not use the golden answer or reference IDs to select them. For single-product questions it still falls back to semantic search with `k=1`.

The application call is `answer_question(store, question, k=1, mode="catalog")`. Here we first inspect the candidate answer from the existing real run. A candidate is the version with our proposed improvement.

In [9]:
candidate = next(row for row in load_artifact('candidate_answers', outputs=True)
                 if row['id'] == example_id)
candidate_judge = next(row for row in load_artifact('candidate_judgments')
                       if row['id'] == example_id)

print('Candidate retrieval route:', candidate['route'])
print('Retrieved product IDs:', candidate['retrieved_ids'])
print('\nCANDIDATE ANSWER:', candidate['answer'])

Candidate retrieval route: category
Retrieved product IDs: ['APK-801', 'APK-802', 'APC-901', 'APC-902', 'APW-1001', 'APW-1002']

CANDIDATE ANSWER: The Home Appliances catalog includes:
- BrewCraft Duo (APK-801)
- FreshVault 12 (APK-802)
- CleanSweep R3 (APC-901)
- SteamMate S1 (APC-902)
- AirPure Cube (APW-1001)
- SleepLight Rhythm (APW-1002)


### Optional: run this one experiment yourself

You can understand the saved example without this cell making any calls. To generate fresh results, change `RUN_EXAMPLE` to `True`.

`services/learning.py` runs only Q61 twice: semantic retrieval, then catalog retrieval. It asks the same Gemini judge to score each answer, waits eight seconds between generation requests, and saves a timestamped `learning_*.json` report. It does not run the advanced suite. Fresh scores may differ.

Preparation must already be complete. The root `.env` supplies your Groq and Google keys. After a fresh run, Step 10 compares those new results.

In [10]:
RUN_EXAMPLE = False
if RUN_EXAMPLE:
    from services.learning import run_example
    experiment = run_example(example_id)
    baseline = experiment['versions']['baseline']['answer']
    baseline_judge = experiment['versions']['baseline']['judgment']
    candidate = experiment['versions']['candidate']['answer']
    candidate_judge = experiment['versions']['candidate']['judgment']
else:
    print('Using the saved real example. No new model calls.')

Using the saved real example. No new model calls.


## 10. Compare and explain the result

Compare the same question using the same reference and judging method. We focus on completeness; correctness is shown so we can check whether adding information damaged the stated facts.

The original saved run has completeness **0.17 → 1.0**: one of six required products became all six. Confirm this by reading the answers, not just their scores. A fresh run may produce different scores.

For this teaching example, keep the category retrieval change. One example does not establish that the change improves every possible question. The earlier all-question retrieval results provide additional evidence, while unseen questions and reviewed references are still needed.

In [11]:
comparison = pd.DataFrame([
    {'version': 'baseline', 'retrieved_products': len(baseline['retrieved_ids']),
     'correctness': baseline_judge['data']['correctness'],
     'completeness': baseline_judge['data']['completeness']},
    {'version': 'candidate', 'retrieved_products': len(candidate['retrieved_ids']),
     'correctness': candidate_judge['data']['correctness'],
     'completeness': candidate_judge['data']['completeness']}
])
display(comparison)
print('Completeness change:', round(comparison.iloc[1]['completeness']
                                    - comparison.iloc[0]['completeness'], 2))

,version,retrieved_products,correctness,completeness
0,baseline,1,1.0,0.17
1,candidate,6,1.0,1.00


Completeness change: 0.83


### Finish here

You have completed the basic evaluation loop when you can explain:

1. Why the first answer was correct but incomplete.
2. Which retrieval setting changed and how that changed the available evidence.
3. Why we compared the same question, reference, generator and judge.
4. Why reading actual answers still matters after seeing a high score.

[The project README](readme.md) explains the idea, setup, source documents, goldens and folder structure. Finish this basic loop before introducing any additional evaluation topic.